# Exercício 1.3 — Pipeline de RAG com Ferramentas Open-Source

**Objetivo:** Construir um pipeline de RAG que ingira os documentos da NovaTech (`.md`), gere embeddings, armazene no ChromaDB, e responda perguntas citando fontes.

**Stack:** Python · sentence-transformers (`all-MiniLM-L6-v2`) · ChromaDB · LangChain · Groq (LLaMA 3.1)

**System Prompt:** reutilizado do Exercício 1.2 (v2), com a seção de chunks injetada dinamicamente pelo RAG.

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain.tools import tool
from langchain.agents import create_agent


/var/folders/11/g35g9fxj3r30h96fl9nfpfp40000gp/T/ipykernel_14720/204621984.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
import os

In [ ]:
os.environ["GROQ_API_KEY"] = ""

In [4]:
# Mesmos modelos do pipeline original — embeddings open-source + LLM via Groq
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=os.environ["GROQ_API_KEY"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
import os

DOCS_PATH = "../dados"

# Os 5 arquivos obrigatórios da NovaTech
REQUIRED_FILES = [
    ("POL-001",      f"{DOCS_PATH}/POL-001-politica-devolucao.md"),
    ("PROC-042-v1",  f"{DOCS_PATH}/PROC-042-frete-especial-v1.md"),
    ("PROC-042-v2",  f"{DOCS_PATH}/PROC-042-v2-frete-especial-revisado.md"),
    ("SLA-2024",     f"{DOCS_PATH}/SLA-2024-tabela-sla-clientes.md"),
    ("FAQ",          f"{DOCS_PATH}/FAQ-atendimento.md"),
]

# Validação: verifica se todos os arquivos existem antes de prosseguir
print("── Validação de arquivos ──────────────────────────────")
missing = []
for doc_id, path in REQUIRED_FILES:
    exists = os.path.exists(path)
    status = "✓" if exists else "✗ AUSENTE"
    print(f"  {status}  {doc_id:15s}  {path}")
    if not exists:
        missing.append(path)

if missing:
    raise FileNotFoundError(
        f"\n{len(missing)} arquivo(s) não encontrado(s):\n" + "\n".join(missing)
    )
print(f"\nTodos os {len(REQUIRED_FILES)} arquivos validados com sucesso.\n")

# Ingestão — cada arquivo .md é um documento independente
all_docs = []
for doc_id, path in REQUIRED_FILES:
    from langchain_community.document_loaders import TextLoader
    loader = TextLoader(path, encoding="utf-8")
    docs = loader.load()
    for doc in docs:
        doc.metadata["source"] = doc_id
    all_docs.extend(docs)
    print(f"✓ Carregado: {doc_id:15s} — {len(docs[0].page_content):>5} chars")

print(f"\nTotal de documentos carregados: {len(all_docs)}")


── Validação de arquivos ──────────────────────────────
  ✓  POL-001          ../dados/POL-001-politica-devolucao.md
  ✓  PROC-042-v1      ../dados/PROC-042-frete-especial-v1.md
  ✓  PROC-042-v2      ../dados/PROC-042-v2-frete-especial-revisado.md
  ✓  SLA-2024         ../dados/SLA-2024-tabela-sla-clientes.md
  ✓  FAQ              ../dados/FAQ-atendimento.md

Todos os 5 arquivos validados com sucesso.

✓ Carregado: POL-001         —  3275 chars
✓ Carregado: PROC-042-v1     —  1551 chars
✓ Carregado: PROC-042-v2     —  2225 chars
✓ Carregado: SLA-2024        —  2663 chars
✓ Carregado: FAQ             —  3877 chars

Total de documentos carregados: 5


In [6]:
# Classificação de autoridade por documento — definida na ingestão, lida no retrieval.
# doc_type  : "normativo" | "informal"
# doc_authority: 1 (mais autoritativo) → 5 (menos autoritativo)
#
# PROC-042-v1 é classificado como "normativo" com autoridade mínima (5) porque
# o sistema da NovaTech não possui indicação formal de vigência ou obsolescência.
# A preferência por v2 é garantida pelo reranking (authority=1 vs 5), não por
# uma flag de "desatualizado" que o sistema não sustenta.
DOC_METADATA = {
    "POL-001":      {"doc_type": "normativo", "doc_authority": 2},
    "PROC-042-v2":  {"doc_type": "normativo", "doc_authority": 1},
    "SLA-2024":     {"doc_type": "normativo", "doc_authority": 3},
    "PROC-042-v1":  {"doc_type": "normativo", "doc_authority": 5},
    "FAQ":          {"doc_type": "informal",  "doc_authority": 4},
}

# ── Estágio 1: divide nos cabeçalhos Markdown ────────────────────────────────
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#",    "Header 1"),
        ("##",   "Header 2"),
        ("###",  "Header 3"),
        ("####", "Header 4"),
    ],
    strip_headers=False,
)

# ── Estágio 2: subdividir seções longas ───────────────────────────────────────
# chunk_size=1000 chars ≈ 750 tokens — faixa de 400-800 tokens recomendada em
# exercicio-1-1.md (seção 4.1) para documentos com tabelas, mantendo-as atômicas.
# chunk_overlap=67 chars ≈ 50 tokens — preserva continuidade sem inflar o índice.
sub_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=1000,
    chunk_overlap=67,
    length_function=len,
    add_start_index=True,
)

chunks = []
for doc in all_docs:
    md_chunks = md_splitter.split_text(doc.page_content)
    sub_chunks = sub_splitter.split_documents(md_chunks)
    for chunk in sub_chunks:
        source = doc.metadata.get("source", "Desconhecido")
        chunk.metadata["source"] = source

        # Injeta classificação de autoridade no metadado do chunk
        meta = DOC_METADATA.get(source, {"doc_type": "informal", "doc_authority": 4})
        chunk.metadata["doc_type"]      = meta["doc_type"]
        chunk.metadata["doc_authority"] = meta["doc_authority"]

        # ── Contextual prefix ──────────────────────────────────────────────
        # Injeta [Fonte | hierarquia de seção] no INÍCIO do texto do chunk
        # antes de gerar o embedding. O modelo encoda "PROC-042-v2 Fórmula de
        # cálculo Multiplicadores regionais" junto com o conteúdo — resolvendo
        # o mismatch semântico para queries em português sobre tabelas técnicas.
        header_path = " > ".join(
            v for k, v in sorted(chunk.metadata.items()) if k.startswith("Header")
        )
        prefix = f"[{source}] {header_path}\n" if header_path else f"[{source}]\n"
        chunk.page_content = prefix + chunk.page_content
    chunks.extend(sub_chunks)

print(f"Total de chunks gerados: {len(chunks)}")

for i, c in enumerate(chunks[:3]):
    headers = {k: v for k, v in c.metadata.items() if k.startswith("Header")}
    print(f"\n--- Chunk {i+1} | fonte: {c.metadata.get('source')} | tipo: {c.metadata.get('doc_type')} | autoridade: {c.metadata.get('doc_authority')} ---")
    print(c.page_content[:300])


Total de chunks gerados: 37

--- Chunk 1 | fonte: POL-001 | tipo: normativo | autoridade: 2 ---
[POL-001] POL-001 — Política de Devolução de Mercadorias
# POL-001 — Política de Devolução de Mercadorias  
**Versão:** 3.1
**Última atualização:** 15/01/2024
**Responsável:** Diretoria de Operações
**Classificação:** Documento normativo — uso obrigatório pelo time de atendimento

--- Chunk 2 | fonte: POL-001 | tipo: normativo | autoridade: 2 ---
[POL-001] POL-001 — Política de Devolução de Mercadorias > 1. Objetivo
## 1. Objetivo  
Esta política define as regras e procedimentos para devolução de mercadorias transportadas pela NovaTech, aplicável a todos os tipos de cliente e categorias de carga, salvo exceções explicitamente listadas na seç

--- Chunk 3 | fonte: POL-001 | tipo: normativo | autoridade: 2 ---
[POL-001] POL-001 — Política de Devolução de Mercadorias > 2. Escopo
## 2. Escopo  
Aplica-se a todas as devoluções solicitadas por clientes da NovaTech após a entrega da mercadoria. Não

In [7]:
import chromadb

# CWD do kernel é /dgs-ai-first — o notebook fica em /dgs-ai-first/jupyter/
NOTEBOOK_DIR = os.path.join(os.getcwd(), "jupyter")
PERSIST_DIR = os.path.join(NOTEBOOK_DIR, "rag_novatech_index")

# Usa PersistentClient diretamente e deleta a collection pelo nome antes de recriar.
# Isso fecha a conexão SQLite de forma limpa — sem shutil.rmtree, sem readonly error.
# O shutil.rmtree é problemático em notebooks porque o ChromaDB mantém connection
# pooling interno que não é liberado por del/gc.collect().
chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)
try:
    chroma_client.delete_collection("langchain")
    print("Collection anterior removida.")
except Exception:
    pass  # não existia ainda

db = Chroma.from_documents(
    chunks,
    embedding=embedding_model,
    client=chroma_client,
    collection_name="langchain",
)
print(f"ChromaDB populado: {db._collection.count()} chunks indexados em '{PERSIST_DIR}'")


ChromaDB populado: 37 chunks indexados em '/Users/thaynar.lima/Documents/dgs-ai-first/jupyter/jupyter/rag_novatech_index'


## Etapa 2 — Busca por Similaridade

Função que recebe uma pergunta, gera o embedding via `all-MiniLM-L6-v2`, busca os N chunks mais próximos no ChromaDB e retorna os resultados **com score de similaridade** (distância L2 — quanto menor, mais similar).

In [8]:
vectordb = Chroma(persist_directory=PERSIST_DIR, embedding_function=embedding_model)

def search_docs(query: str, k: int = 6):
    """Retrieval com reranking por autoridade de documento.

    Fluxo:
    1. Busca k*4 candidatos no ChromaDB excluindo doc_type "informal" e
       "desatualizado" — FAQ e versões antigas nunca entram no candidate set.
    2. Fallback sem filtro se retornar menos de k resultados.
    3. Deduplica por hash de conteúdo.
    4. Reranking combinado: 60% score L2 + 40% doc_authority do metadado.
       A autoridade vem da ingestão — não de lookup em runtime.
    """
    where = {"doc_type": {"$eq": "normativo"}}
    raw = vectordb.similarity_search_with_score(query, k=k * 4, filter=where)

    if len(raw) < k:
        raw = vectordb.similarity_search_with_score(query, k=k * 4)

    seen: set = set()
    deduped = []
    for doc, score in raw:
        key = " ".join(doc.page_content.split())
        if key not in seen:
            seen.add(key)
            deduped.append((doc, score))

    if deduped:
        scores = [s for _, s in deduped]
        min_s, max_s = min(scores), max(scores)
        score_range = (max_s - min_s) or 1.0

        def rank_key(item):
            doc, score = item
            authority      = doc.metadata.get("doc_authority", 4)
            norm_score     = (score - min_s) / score_range
            norm_authority = authority / 5
            return 0.6 * norm_score + 0.4 * norm_authority

        deduped.sort(key=rank_key)

    return deduped[:k]

# Teste rápido da busca
print("Teste de busca: 'prazo de devolução'")
for doc, score in search_docs("prazo de devolução"):
    print(f"  [{doc.metadata.get('source')}] tipo={doc.metadata.get('doc_type')} | score={score:.4f} | {doc.page_content[:80]}...")


Teste de busca: 'prazo de devolução'
  [POL-001] tipo=normativo | score=0.7180 | [POL-001] POL-001 — Política de Devolução de Mercadorias > 3. Regras de Devoluçã...
  [POL-001] tipo=normativo | score=0.9956 | [POL-001] POL-001 — Política de Devolução de Mercadorias > 3. Regras de Devoluçã...
  [POL-001] tipo=normativo | score=1.0814 | [POL-001] POL-001 — Política de Devolução de Mercadorias > 3. Regras de Devoluçã...
  [POL-001] tipo=normativo | score=1.1328 | [POL-001] POL-001 — Política de Devolução de Mercadorias > 3. Regras de Devoluçã...
  [PROC-042-v2] tipo=normativo | score=1.2631 | [PROC-042-v2] PROC-042-v2 — Procedimento de Cálculo de Frete Especial (Revisado)...
  [PROC-042-v2] tipo=normativo | score=1.3333 | [PROC-042-v2] PROC-042-v2 — Procedimento de Cálculo de Frete Especial (Revisado)...


## Etapa 3 — Montagem de Prompt

O system prompt reutiliza o **v2 do Exercício 1.2** (partes estáticas: identidade, regras, prioridade de fontes, formato). 
A seção `DOCUMENTAÇÃO DISPONÍVEL NESTA CONSULTA` é **dinâmica** — preenchida a cada query com os chunks recuperados pelo RAG.

In [9]:
# ── SYSTEM PROMPT v2 — exercicio-1-2.md (partes estáticas: ~290 tokens) ──────
# Fonte: System Prompt v2 de exercicio-1-2.md, seção "System Prompt v2"
# A seção DOCUMENTAÇÃO DISPONÍVEL é injetada dinamicamente pelo RAG (assemble_prompt).
SYSTEM_PROMPT_STATIC = """## IDENTIDADE
Você é NovaTech Assist, assistente de IA da equipe de atendimento ao cliente da NovaTech 
Logística. Seu papel é ajudar os atendentes a encontrar respostas rápidas e precisas com 
base na documentação oficial da empresa.
Você apoia atendentes internos — não atende clientes diretamente.

## REGRAS OBRIGATÓRIAS
1. Cite sempre a fonte: nome do documento e seção (ex: "POL-001, seção 3.2").
2. Nunca invente prazos, valores numéricos ou multiplicadores. Use apenas os dados 
   presentes nos documentos fornecidos abaixo.
3. Quando a informação não estiver nos documentos disponíveis, diga explicitamente: 
   "Não encontrei essa informação na documentação disponível. Recomendo escalar para 
   o supervisor ou consultar a área responsável."
4. Responda em português formal, porém direto e acessível.

## PRIORIDADE DE FONTES
Quando houver informações conflitantes entre documentos, siga esta ordem:
1. Documentos normativos mais recentes (versão mais nova prevalece)
2. Políticas formais (POL-001, SLA-2024)
3. FAQ (use apenas para orientação de processo — nunca como fonte de regras ou valores)

## FORMATO DE RESPOSTA
- Resposta direta à pergunta (1 a 3 frases)
- Fonte: cite o documento e a seção
- Ação recomendada: indique o próximo passo do atendente, quando aplicável"""

# ── PARTE DINÂMICA: chunks + pergunta (varia a cada query) ───────────────────
def assemble_prompt(question: str, retrieved_chunks) -> str:
    """Monta o prompt completo: system prompt estático + chunks dinâmicos + pergunta."""
    chunk_blocks = []
    for i, (doc, score) in enumerate(retrieved_chunks):
        fonte = doc.metadata.get('source', 'Desconhecido')
        chunk_blocks.append(
            f"[Chunk {i+1} — {fonte} | similaridade: {score:.4f}]\n{doc.page_content}"
        )
    chunks_section = "\n\n".join(chunk_blocks)
    
    return (
        f"{SYSTEM_PROMPT_STATIC}\n\n"
        f"## DOCUMENTAÇÃO DISPONÍVEL NESTA CONSULTA\n\n"
        f"{chunks_section}\n\n"
        f"---\nPergunta do atendente: {question}"
    )

print("Funções de montagem de prompt carregadas.")


Funções de montagem de prompt carregadas.


## Etapa 4 — Agente com Tool de Recuperação

In [10]:
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Recupera trechos relevantes da documentação NovaTech para responder a pergunta."""
    results = search_docs(query, k=3)
    docs = [doc for doc, _ in results]
    serialized = "\n\n".join(doc.page_content for doc in docs)
    return serialized, docs


agent = create_agent(llm, [retrieve_context], system_prompt=SYSTEM_PROMPT_STATIC)

print("Agente criado com tool de recuperação NovaTech.")

Agente criado com tool de recuperação NovaTech.


In [11]:
def ask(question):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    context = next(
        (msg.artifact for msg in result["messages"] if hasattr(msg, "artifact")),
        None
    )
    answer = result["messages"][-1].content
    return answer, context

## Etapa 5 — Testes: Mapa de Cobertura do Anexo B

5 perguntas do gabarito do Anexo B. Para cada uma:
- Chunks recuperados + score
- Comparação com chunks esperados (gabarito)
- Prompt completo montado (pronto para colar no Claude)

In [12]:
# Gabarito alinhado ao Anexo B — Mapa de cobertura + Armadilhas
# chunks_obrigatorios : devem aparecer no top-k (DEVEM ser recuperados)
# chunks_aceitaveis   : podem aparecer com relevância menor (OK se aparecerem)
# chunks_armadilha    : se aparecerem, sinalizam um problema conhecido (Anexo B, seção Armadilhas)
GABARITO = [
    {
        "pergunta": "Qual o prazo de devolução?",
        "chunks_obrigatorios": ["POL-001"],   # POL-001-A (seção 3.1) e POL-001-B (seção 3.2)
        "chunks_aceitaveis":   ["POL-001"],   # POL-001-C (procedimento)
        "chunks_armadilha":    [],
        "descricao": "Prazo geral + exceções",
    },
    {
        "pergunta": "Posso devolver carga perigosa?",
        "chunks_obrigatorios": ["POL-001"],   # POL-001-B (seção 3.2 — cargas perigosas)
        "chunks_aceitaveis":   ["FAQ", "POL-001"],
        "chunks_armadilha":    ["FAQ"],       # Armadilha 2: FAQ como fonte para regra crítica
        "descricao": "Exceção ao prazo padrão — NÃO elegível pelo processo padrão",
    },
    {
        "pergunta": "Qual o SLA do cliente Gold?",
        "chunks_obrigatorios": ["SLA-2024"],  # SLA-2024-B (seção 2 — chamados gerais)
        "chunks_aceitaveis":   ["SLA-2024", "FAQ"],
        "chunks_armadilha":    [],
        "descricao": "SLA tier Gold",
    },
    {
        "pergunta": "Frete para 600kg para Manaus?",
        "chunks_obrigatorios": ["PROC-042-v2"],  # PROC-042v2-B (multiplicadores) + PROC-042v2-A (fórmula)
        "chunks_aceitaveis":   ["PROC-042-v2"],
        "chunks_armadilha":    ["PROC-042-v1"],  # Armadilha 1: v1 e v2 coexistem — multiplicadores divergem
        "descricao": "Fórmula + multiplicador Norte — v2 prevalece sobre v1",
    },
    {
        "pergunta": "Qual o multiplicador para o Sudeste?",
        "chunks_obrigatorios": ["PROC-042-v2"],  # PROC-042v2-B (seção 2.1 — v2: Sudeste=1.1)
        "chunks_aceitaveis":   ["PROC-042-v2"],
        "chunks_armadilha":    ["PROC-042-v1"],  # Armadilha 1: v1 tem Sudeste=1.0 — contradição direta
        "descricao": "Multiplicador regional Sudeste — risco contradição v1 (1.0) vs v2 (1.1)",
    },
]

for idx, caso in enumerate(GABARITO, 1):
    pergunta = caso["pergunta"]
    obrigatorios  = caso["chunks_obrigatorios"]
    armadilhas    = caso["chunks_armadilha"]

    print(f"{'='*65}")
    print(f"Teste {idx}: {pergunta}")
    print(f"Descrição: {caso['descricao']}")
    print(f"Esperados:  {obrigatorios}  |  Armadilhas: {armadilhas or '—'}")

    resultados = search_docs(pergunta, k=3)

    acertou     = False
    tem_armadilha = False
    print("\nChunks recuperados:")
    for doc, score in resultados:
        fonte = doc.metadata.get("source", "?")
        é_obrigatorio = any(e in fonte for e in obrigatorios)
        é_armadilha   = any(a in fonte for a in armadilhas)

        if é_obrigatorio:
            acertou = True
        if é_armadilha:
            tem_armadilha = True

        flag = "✓" if é_obrigatorio else ("⚠ ARMADILHA" if é_armadilha else "✗")
        print(f"  {flag:12s} [{fonte}] score={score:.4f} | {doc.page_content[:90].strip()}...")

    status = "✓ OK" if acertou else "✗ FALHOU"
    aviso  = "  ⚠  Armadilha ativa — chunk de versão/fonte incorreta recuperado" if tem_armadilha else ""
    print(f"\nResultado: {status}{aviso}")
    print()


Teste 1: Qual o prazo de devolução?
Descrição: Prazo geral + exceções
Esperados:  ['POL-001']  |  Armadilhas: —

Chunks recuperados:
  ✓            [POL-001] score=0.7070 | [POL-001] POL-001 — Política de Devolução de Mercadorias > 3. Regras de Devolução > 3.5. C...
  ✓            [POL-001] score=0.9600 | [POL-001] POL-001 — Política de Devolução de Mercadorias > 3. Regras de Devolução > 3.3. P...
  ✓            [POL-001] score=1.1206 | [POL-001] POL-001 — Política de Devolução de Mercadorias > 3. Regras de Devolução > 3.4. D...

Resultado: ✓ OK

Teste 2: Posso devolver carga perigosa?
Descrição: Exceção ao prazo padrão — NÃO elegível pelo processo padrão
Esperados:  ['POL-001']  |  Armadilhas: ['FAQ']

Chunks recuperados:
  ✓            [POL-001] score=1.0117 | [POL-001] POL-001 — Política de Devolução de Mercadorias > 3. Regras de Devolução > 3.5. C...
  ✓            [POL-001] score=1.1031 | [POL-001] POL-001 — Política de Devolução de Mercadorias > 3. Regras de Devolução > 3.2. E...

In [13]:
# Exibe o prompt completo montado para cada teste
# Cole no Claude para obter a resposta e avaliar guardrails

for idx, caso in enumerate(GABARITO, 1):
    pergunta = caso["pergunta"]
    resultados = search_docs(pergunta, k=3)
    prompt = assemble_prompt(pergunta, resultados)
    
    print(f"{'='*65}")
    print(f"PROMPT COMPLETO — Teste {idx}: {pergunta}")
    print(f"{'='*65}")
    print(prompt)
    print()

PROMPT COMPLETO — Teste 1: Qual o prazo de devolução?
## IDENTIDADE
Você é NovaTech Assist, assistente de IA da equipe de atendimento ao cliente da NovaTech 
Logística. Seu papel é ajudar os atendentes a encontrar respostas rápidas e precisas com 
base na documentação oficial da empresa.
Você apoia atendentes internos — não atende clientes diretamente.

## REGRAS OBRIGATÓRIAS
1. Cite sempre a fonte: nome do documento e seção (ex: "POL-001, seção 3.2").
2. Nunca invente prazos, valores numéricos ou multiplicadores. Use apenas os dados 
   presentes nos documentos fornecidos abaixo.
3. Quando a informação não estiver nos documentos disponíveis, diga explicitamente: 
   "Não encontrei essa informação na documentação disponível. Recomendo escalar para 
   o supervisor ou consultar a área responsável."
4. Responda em português formal, porém direto e acessível.

## PRIORIDADE DE FONTES
Quando houver informações conflitantes entre documentos, siga esta ordem:
1. Documentos normativos mais rec

## Etapa 6 — Respostas via Agente (LLM local via Groq)

Aqui o agente chama automaticamente a tool de recuperação e gera a resposta.
Avalie: a resposta está correta? Citou a fonte? Respeitou os guardrails?

In [14]:
for idx, caso in enumerate(GABARITO, 1):
    pergunta = caso["pergunta"]
    print(f"{'='*65}")
    print(f"Teste {idx}: {pergunta}")
    resposta, _ = ask(pergunta)
    print(f"Resposta:\n{resposta}")
    print()

Teste 1: Qual o prazo de devolução?


Resposta:
O prazo de devolução é de até 7 (sete) dias úteis após a data de recebimento confirmada no sistema de tracking. A contagem de dias úteis exclui sábados, domingos e feriados nacionais. [POL-001, seção 3.1]

Teste 2: Posso devolver carga perigosa?


Resposta:
Sim, é possível devolver carga perigosa, mas somente após comunicação prévia com o setor de Gestão de Riscos (ramal 4500) para tratamento individual. Além disso, é importante verificar se a carga está classificada nas classes 1 a 6 da ANTT e se há aprovação prévia do gerente de operações regional caso o peso seja maior que 5.000kg.

Fonte: [POL-001] POL-001 — Política de Devolução de Mercadorias, seção 3.2 e 3.5, e [PROC-042-v1] PROC-042 — Procedimento de Cálculo de Frete Especial, seção 4.

Ação recomendada: o atendente deve entrar em contato com o setor de Gestão de Riscos (ramal 4500) para agendar a devolução e obter a aprovação necessária. Além disso, é importante verificar se a carga está classificada nas classes 1 a 6 da ANTT e se há aprovação prévia do gerente de operações regional caso o peso seja maior que 5.000kg.

Teste 3: Qual o SLA do cliente Gold?


Resposta:
O SLA do cliente Gold é de 99,9% para chamados gerais e 100% para incidentes críticos, medido pelo sistema de chamados (Azure DevOps) a partir do timestamp de abertura do chamado. [SLA-2024, seção 5. Medição e reportes]

Teste 4: Frete para 600kg para Manaus?


Resposta:
O cálculo do frete para 600kg para Manaus depende do fator de peso, que é calculado com base no peso da carga. Segundo a documentação, o fator de peso é de 1,15 para cargas de 1.001kg a 3.000kg. Portanto, o fator de peso para 600kg é 1,15.

O próximo passo é calcular o valor do frete com base na fórmula de cálculo do procedimento PROC-042-v2. Para isso, é necessário saber o valor base da tarifa publicada na tabela mensal de fretes e o multiplicador regional para a região de destino, que é a Região Norte, conforme a seção 2.1 do procedimento.

Recomendo consultar a tabela mensal de fretes e o multiplicador regional para a Região Norte para calcular o valor do frete.

Fonte: PROC-042-v2, seção 2 e 4.

Ação recomendada: Consultar a tabela mensal de fretes e o multiplicador regional para a Região Norte.

Teste 5: Qual o multiplicador para o Sudeste?


Resposta:
O multiplicador para o Sudeste é de 1.1, de acordo com o [PROC-042-v2] PROC-042-v2 — Procedimento de Cálculo de Frete Especial (Revisado).



In [ ]:
# Pergunta interativa — mesmo padrão do notebook original
user_question = input("Atendente: ")
resposta, docs = ask(user_question)
print("\nNovaTech Assist:", resposta)
if docs:
    print("\nFontes consultadas:")
    for d in docs:
        print(f"  [{d.metadata.get('source')}] {d.page_content[:80]}...")

---

# Relatório — Exercício 1.3

## Stack utilizada

| Componente | Biblioteca | Função |
|-----------|-----------|--------|
| Embeddings | `sentence-transformers/all-MiniLM-L6-v2` | Geração de embeddings open-source |
| Vector store | `ChromaDB` | Armazenamento e busca por similaridade |
| LLM | `Groq / LLaMA 3.1-8b-instant` | Geração de respostas |
| Orquestração | `LangChain + LangGraph` | Chunking, loaders, agente |
| Splitter | `MarkdownHeaderTextSplitter` + `RecursiveCharacterTextSplitter` | Chunking em dois estágios |

---

## Evidência de uso do GitHub Copilot

O GitHub Copilot (modo Agent, VS Code) foi utilizado para atualizar os parâmetros do pipeline após a análise do exercício 1.1. A sessão abaixo mostra o Copilot aplicando 5 mudanças no notebook com justificativas rastreadas até os documentos de referência.

**Prompt dado ao Copilot:**
> "Now update the system prompt cell to match exactly the v2 from exercicio-1-2.md"

**Resultado (5/5 tarefas concluídas):**

![Copilot Chat - Atualizar Pipeline de RAG](./copilot-evidencia-1.png)

**Mudanças aplicadas e aceitas:**

| Parâmetro | Antes | Depois | Justificativa (exercicio-1-1.md) |
|-----------|-------|--------|----------------------------------|
| `chunk_size` | 600 chars | 1000 chars (≈750 tokens) | Seção 4.1: até 800 tokens para preservar tabelas intactas (PROC-042) |
| `chunk_overlap` | 80 chars | 67 chars (≈50 tokens) | Seção 4.1: 50 tokens de overlap para continuidade entre seções |
| `k` (search) | 3 | 6 | Seção 3: sweet spot de 6-10 chunks evita *lost in the middle* |
| System prompt | Versão aproximada | v2 exata do exercício 1.2 | Guardrails completos: prioridade de fontes, citação obrigatória, fallback explícito |

**O que foi ajustado manualmente após a sugestão:** o reranking por autoridade de documento e o filtro por `doc_type` foram adicionados depois, pois o Copilot não tinha contexto dos problemas de retrieval identificados nos testes.

---

## O que o `create_agent` faz por baixo dos panos

`create_agent(llm, tools, system_prompt)` é um wrapper do LangGraph que monta um **StateGraph de dois nós** e retorna um executável compilado:

1. **Serialização das tools:** cada `@tool` vira um JSON Schema (formato *function calling* da OpenAI) enviado junto com cada chamada ao LLM.
2. **Nó `agent` (raciocínio):** o LLM recebe o histórico de mensagens acumulado no estado e decide: responde diretamente (→ `END`) ou emite um `AIMessage` com `tool_calls` (→ nó `tools`).
3. **Nó `tools` (execução):** o `ToolNode` deserializa os argumentos, executa a função Python e adiciona o retorno como `ToolMessage` ao estado.
4. **Loop:** o grafo volta ao nó `agent` com o estado atualizado. O LLM vê o resultado da tool e decide se precisa de mais contexto ou já pode responder.
5. **Parada:** quando o `AIMessage` não contém `tool_calls`, o roteador condicional direciona para `END`.

O `system_prompt` é inserido como `SystemMessage` no início de cada invocação.

---

## Problemas encontrados e correções

### Problema 1 — Conflito de versões PROC-042 v1 vs v2
**Testes afetados:** 4 e 5

Os dois documentos coexistem no índice sem hierarquia de versão. O LLM pode misturar multiplicadores antigos (v1: Norte=1.6, Sudeste=1.0) com os novos (v2: Norte=1.8, Sudeste=1.1).

**Mitigação aplicada:** contextual prefix encoda `[PROC-042-v2]` junto ao conteúdo; `doc_authority` rebaixa PROC-042-v1 (autoridade=5) frente ao v2 (autoridade=1) no reranking — fazendo v2 subir para o topo em `search_docs` e, após a correção do Problema 6, também dentro do agente.

**Limitação residual:** PROC-042-v1 ainda é classificado como `doc_type: normativo`, então entra no candidate set quando o fallback sem filtro é acionado. Para Teste 2, o v1 aparece como 3º chunk porque a query sobre "carga perigosa" semanticamente conecta com a seção 4 do v1 (condições especiais). Correção completa exigiria filtrar versões obsoletas na ingestão.

---

### Problema 2 — Chunking cortava tabelas Markdown no meio ✅ Corrigido
**Observado:** versão inicial com `RecursiveCharacterTextSplitter` único, `chunk_size=600`

O splitter atingia o limite no meio do corpo da tabela de multiplicadores — gerando chunks sem cabeçalho (`| Norte | 1.8 |` isolado). O LLM não conseguia identificar o contexto.

**Correção:** pipeline em dois estágios — `MarkdownHeaderTextSplitter` isola cada seção (incluindo tabelas completas) como chunk atômico; `RecursiveCharacterTextSplitter` subdivide seções longas sem cruzar limites de tabela.

---

### Problema 3 — Mismatch semântico para queries sobre tabelas técnicas ✅ Corrigido
**Observado:** Testes 4 e 5 antes do contextual prefix

`all-MiniLM-L6-v2` é otimizado para inglês — "600kg para Manaus" não conectava semanticamente com "Multiplicadores regionais Norte=1.8".

**Correção:** contextual prefix — cada chunk recebe `[fonte] seção > subseção` no texto antes do embedding. O modelo encoda `"PROC-042-v2 Fórmula de cálculo > Multiplicadores regionais"` junto ao conteúdo.

---

### Problema 4 — Chunks duplicados por reexecução do índice ✅ Corrigido
**Observado:** primeira execução sem limpeza prévia

`Chroma.from_documents()` acrescenta ao índice existente. Reexecuções multiplicavam os chunks, fazendo o mesmo trecho aparecer 3× no top-k.

**Correção:** `chroma_client.delete_collection("langchain")` antes de cada `from_documents()` via `PersistentClient`, sem `shutil.rmtree` (causava erro de conexão SQLite). Deduplicação por hash de conteúdo em `search_docs` como salvaguarda adicional.

---

### Problema 5 — FAQ dominava o top-k em queries de regra/política ✅ Corrigido
**Observado:** Teste 2 — o LLM citava o FAQ como fonte de regra, violando o guardrail de prioridade de fontes.

**Correção:** filtro por `doc_type: normativo` no ChromaDB exclui o FAQ (`doc_type: informal`) do candidate set em `search_docs`. Após a correção do Problema 6 (unificação dos caminhos de retrieval), o filtro agora se aplica também dentro do agente — o FAQ não entra mais no contexto do LLM.

---

### Problema 6 — Desacoplamento entre `search_docs` e `retrieve_context` ✅ Corrigido
**Observado:** `retrieve_context` usava `vectordb.similarity_search` simples — sem filtro por `doc_type` nem reranking por `doc_authority`. Os metadados definidos na ingestão eram ignorados no caminho do agente.

**Efeito:** o agente produzia fontes inventadas, citava FAQ, e usava v1 mesmo após as correções dos Problemas 1 e 5.

**Correção:**
```python
# Antes
docs = vectordb.similarity_search(query, k=3)

# Depois
results = search_docs(query, k=3)
docs = [doc for doc, _ in results]
```

Agora os dois caminhos de retrieval são idênticos — o agente herda filtro, reranking e deduplicação.

---

## Análise crítica dos outputs do agente (Etapa 6)

Etapa 5 (retrieval direto): **5/5 ✓**. Etapa 6 (agente, após correção do Problema 6): **2/5 ✓** — Testes 1 e 5 corretos, Testes 2, 3 e 4 com falhas residuais.

### Teste 1 ⚠️ Parcialmente correto
**Resposta gerada:** "O prazo de devolução é de até 7 (sete) dias úteis após a data de recebimento confirmada no sistema de tracking. [POL-001, seção 3.1]"

Fonte correta (POL-001), mas o valor "7 dias úteis" não corresponde aos três prazos distintos do POL-001 (triagem: 4h; coleta: 2 dias úteis; reembolso: 5 dias úteis). O LLM consolidou os prazos em um único número impreciso — violação da regra 2 (nunca invente valores).

---

### Falha A — Teste 2: PROC-042-v1 citado como fonte complementar ⚠️ Persiste parcialmente
**Resposta gerada:** "Sim, é possível devolver carga perigosa, mas somente após comunicação prévia com Gestão de Riscos. Fonte: POL-001 seção 3.2 e 3.5, e **PROC-042-v1** seção 4."

Melhora: o FAQ não é mais citado ✅. Mas PROC-042-v1 aparece porque `search_docs` para esta query retorna o v1 como 3º chunk (seção 4 do v1 menciona "condições especiais" — match semântico espúrio). O reranking por `doc_authority` não é suficiente para eliminar o v1 quando a query tem baixa similaridade com os chunks de POL-001.

O mérito da resposta ("Sim, é possível") também é questionável — POL-001 seção 3.2 classifica carga perigosa como não elegível pelo processo padrão.

---

### Falha B — Teste 3: seção errada do SLA-2024 ⚠️ Persiste
**Resposta gerada:** "O SLA do cliente Gold é de 99,9% para chamados gerais e 100% para incidentes críticos. [SLA-2024, seção 5. Medição e reportes]"

Fonte correta (SLA-2024) ✅, mas seção errada: os valores de disponibilidade (99,9% / 100%) vêm da seção 5 (medição), não da seção 2 (tabela de SLA por tier). O chunk da seção 2 — que contém os prazos de atendimento do tier Gold — não entrou no top-k porque o score de similaridade de "SLA cliente Gold" priorizou seções com maior densidade do termo.

---

### Falha C — Teste 4: fator de peso aplicado à faixa errada ⚠️ Persiste
**Resposta gerada:** "O fator de peso é de 1,15 para cargas de 1.001kg a 3.000kg. Fonte: PROC-042-v2, seção 2 e 4."

Fonte correta (PROC-042-v2) ✅. Mas o fator 1,15 se aplica à faixa 1.001–3.000kg — para 600kg (abaixo de 1.001kg) o fator correto é 1,0. O LLM aplicou a faixa mais próxima disponível no chunk sem verificar se o peso da query se enquadrava nessa faixa.

---

### Teste 5 ✅ Correto
**Resposta gerada:** "O multiplicador para o Sudeste é de 1.1, de acordo com PROC-042-v2, seção 2.1."

Valor correto, versão correta, citação completa.

---

### Conclusão

A unificação dos caminhos de retrieval (Problema 6) eliminou as falhas mais graves: fontes inventadas, FAQ como fonte normativa, e citação da versão errada do procedimento. As falhas residuais (Testes 1, 2, 3 e 4) têm causas distintas — imprecisão de síntese do LLM, match semântico espúrio de v1 para queries de política, e ausência da seção correta no top-k — e requerem ajustes além do retrieval: query expansion, filtro de versão na ingestão, ou aumento de k com pós-filtragem por seção.

---

## Evidência de uso do GitHub Copilot (Copilot Chat — modo Agent)

O GitHub Copilot foi utilizado no modo **Agent** (VS Code) para atualizar os parâmetros do pipeline diretamente no notebook, com base nas análises documentadas no exercício 1.1.

**Sessão registrada:** tarefa "ATUALIZAR PIPELINE DE RAG" — 5/5 subtarefas concluídas.

![Copilot Chat — Atualizar Pipeline de RAG](./copilot-evidencia-1.png)
